# PRE-PROCESSING:


In [1]:
import json
import time
import requests
import pandas as pd
from pathlib import Path

## Load data

In [2]:
path = '../data/wyndham_smartbin_filllevel.json'

with open(path, 'r') as f:
    json_object = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '../data/wyndham_smartbin_filllevel.json'

## Isolate the data we want to use

In [ ]:
rows = []
for feature in json_object["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    rows.append({
        "timestamp": props["timestamp"],
        "fullnessthreshold": props["fullnessThreshold"],
        "serialnumber": props["serialNumber"],
        "latestfullness": props["latestFullness"],
        "coordinates": coords
    })

df = pd.DataFrame(rows)

### <span style="color:green">What we have so far:</span>

In [ ]:
display(df.head())

## Drop outlier bins

In [ ]:
df = df[~df["serialnumber"].astype(str).isin(["1511202"])]
# bin '1511202' has only 181 samples while all the others have 976 samples 

## A function that disassembles the timestamp into features we can use

<span style="color:orange">!! Several of them might be useless  !!</span>

In [ ]:
def add_time_features(df, ts_col="timestamp", hemisphere="southern", tz=None):
    out = df.copy()

    # Parse to datetime
    ts = pd.to_datetime(out[ts_col])

    # Timezone handling
    if tz is not None:
        if ts.dt.tz is None:
            ts = ts.dt.tz_localize(tz)
        else:
            ts = ts.dt.tz_convert(tz)

    # ISO calendar parts (week/year/week-day)
    iso = ts.dt.isocalendar()

    # Base fields
    out["year"] = ts.dt.year
    out["quarter"] = ts.dt.quarter
    out["month"] = ts.dt.month
    out["week"] = iso.week.astype("Int64")
    out["day_of_year"] = ts.dt.dayofyear
    out["day_of_week"] = ts.dt.weekday      # 0=Mon ... 6=Sun

    # Flags
    # don't know if they are of any use
    out["is_weekend"] = out["day_of_week"].isin([5, 6])
    out["is_month_start"] = ts.dt.is_month_start
    out["is_month_end"] = ts.dt.is_month_end
    out["is_quarter_start"] = ts.dt.is_quarter_start
    out["is_quarter_end"] = ts.dt.is_quarter_end
    out["is_year_start"] = ts.dt.is_year_start
    out["is_year_end"] = ts.dt.is_year_end

    # Season mapping
    # We can argue that this is a "gerneralization" step?
    def _season(month, hemi):
        # Spring = 1, Summer = 2, Autumn = 3, Winter = 4
        if hemi == "southern":
            mapping = {12:2,1:2,2:2,
                       3:3,4:3,5:3,
                       6:4,7:4,8:4,
                       9:1,10:1,11:1}
        else:
            mapping = {12:4,1:4,2:4,
                       3:1,4:1,5:1,
                       6:2,7:2,8:2,
                       9:3,10:3,11:3}
        return mapping.get(month, pd.NA)

    out["season"] = out["month"].apply(lambda m: _season(m, hemisphere))

    return out


nice to have would be to do cyclic encoding on the temporal features

## Add time features and display

In [ ]:
df = add_time_features(df = df, ts_col="timestamp", hemisphere="northern", tz="Australia/Melbourne")
df.head()

## Disassemble "coordinates" feature into lat and lon and add them to the dataframe

In [ ]:
df["lon"] = df["coordinates"].apply(lambda x: x[0])
df["lat"] = df["coordinates"].apply(lambda x: x[1])
df.head()

## Here we can see that there is something weird going on. There should be 33 entries per timestamp but thats not the case. Some timestamps have hundereds of entries. Some of them have only 32


In [ ]:
timestamp_counts = df['timestamp'].value_counts()
print("Unique timestamps and their counts:")
display(timestamp_counts.head(10))

#Uncomment if you want to check how many bins there are and how many samples are present
# print("Unique Serialnumbers:")
# print(df["serialnumber"].unique())
# print(df["serialnumber"].value_counts())
# len(df["serialnumber"].unique())

## <span style="color:green">Gained Knowledge:</span> 
- there are 33 unique serialnumbers
- bin '1511202' has only 181 samples while all the others have 976 samples (drop it?)
- e.g timestamp '2019-01-24' is present 416 times. 32*16 equals 416??. So on this day the bins have been sampled 13 times?

## Next step:
Build a df that is nicer for sending requests\
This is just so the code is easier to understand\

In [ ]:
# a function that returns a df which we want to use for querying the weather API
def unique_serial_coords(df):
    return (
        df[["serialnumber", "lon", "lat"]]
        .dropna(subset=["serialnumber", "lon", "lat"])
        .drop_duplicates(subset=["serialnumber"])
        .reset_index(drop=True)
    )

### Get first and last day of the dataset

In [ ]:
query_timespan = (df["timestamp"].min(),df["timestamp"].max())
# should be ('2018-06-26', '2021-05-03')
print("TimeSpan:")
print(query_timespan)

In [ ]:
query_df = unique_serial_coords(df)

print("Shape of dataframe. Note: there are 33 smart-bins. But only if we didnt drop any bins")
print(query_df.shape)
display(query_df.head())

## Now for every serialnumber we query the weather API from first_day to last_day

- This function takes coordinates and a timerange as input and returns a dataframe with "timestamp", "rain", "sunhours".
- It builds an URL and calls it. You can copy this to your browser and see the format in which it returns the data: 
- https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2024-01-01&end_date=2024-01-07&daily=sunshine_duration,rain_sum,precipitation_hours,daylight_duration&timezone=auto
- This example returns 1 week worth of weatherdata.

In [ ]:

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

def _fetch_open_meteo_daily(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join([
            "sunshine_duration",   # seconds/day
            "rain_sum",            # mm/day
            "precipitation_hours", # hours/day
            "daylight_duration"    # seconds/day
        ]),
        "timezone": "auto"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json() # Parse the JSON response

    # Create a DataFrame with the requested columns
    daily_data = {
        'timestamp': data['daily']['time'],
        'rain': data['daily']['rain_sum'],
        'sunhours': [s / 3600 for s in data['daily']['sunshine_duration']] # Convert seconds to hours
    }
    return pd.DataFrame(daily_data)


# Convert the query_df so it has one column for every timestamp. Now a cell holds the json string return value from the API


- Now we make 33 API calls. This takes a while.
- <span style="color:red"> We call the API only every 3 seconds, because otherwise they block our requests.</span>

-TODO: save data to a tempfile <span style="color:green"> DONE.</span>


In [ ]:
cache_file = Path("../data/weather_cache.json")

# Load existing cache if present
if cache_file.exists():
    print("Loading cached data")
    weather_all = pd.read_json(cache_file)
else:
    weather_all = pd.DataFrame()

weather_frames = [weather_all]
query_df = unique_serial_coords(df)
start_date, end_date = query_timespan

for _, r in query_df.iterrows():
    serial, lat, lon = r["serialnumber"], r["lat"], r["lon"]

    # Check if already cached
    cached_exists = (
        not weather_all.empty and
        ((weather_all["serialnumber"] == serial) &
         (weather_all["lat"] == lat) &
         (weather_all["lon"] == lon)).any()
    )

    if cached_exists:
        print(f"Cache hit {serial}")
        continue

    # Fetch new data if missing
    print(f"Query API for {serial}")
    daily = _fetch_open_meteo_daily(lat, lon, start_date, end_date)
    time.sleep(4)
    daily = daily.assign(serialnumber=serial, lat=lat, lon=lon)
    weather_frames.append(daily)

# Merge + save updated cache
weather_all = pd.concat(weather_frames, ignore_index=True)
weather_all.to_json(cache_file, orient="records")

display(weather_all.head())


## Now merge this dataframe into our original

Make sure "timestamp" has the same datatype in both dataframes 

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
weather_all["timestamp"] = pd.to_datetime(weather_all["timestamp"])

# In case you want to look at the datatypes
# print(df.head().dtypes)
# print(weather_all.head().dtypes)

In [ ]:
df = df.merge(
    weather_all[["timestamp", "serialnumber", "lat", "lon", "rain", "sunhours"]],
    on=["timestamp", "serialnumber", "lat", "lon"],
    how="left"
)

display(df.head(9))

## Augment Data with Geoapify Metadata

In [ ]:
import time
import numpy as np
import pandas as pd
import requests

GEOAPIFY_API_KEY = "4e2560f28d7342989c75bdcd01f90797"
radius_m = 100  # hyperparameter
categories = "commercial,catering"  # shops + restaurants (maybe also take entertainment? nightlife and so)
limit = 500  # limit of api
sleep_s = 0.20  # be nice to API

#calculates distance given coordinates
def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

#returns the about of places in a given radius
def geoapify_count_places(lat, lon, radius_m, categories, api_key, limit=200):
    base_url = "https://api.geoapify.com/v2/places"
    offset = 0
    total = 0

    while True:
        params = {
            "apiKey": api_key,
            "categories": categories,
            "filter": f"circle:{lon},{lat},{int(radius_m)}",  # lon,lat,radiusMeters
            "limit": limit,
            "offset": offset,
        }
        r = requests.get(base_url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        n = len(data.get("features", []))
        total += n

        if n < limit:
            break

        offset += limit
        time.sleep(sleep_s)

    return total

# UNIQUE BIN COORDS (one lat/lon per serialnumber) 
bins = (
    df[["serialnumber", "lat", "lon"]]
    .dropna(subset=["lat", "lon"])
    .assign(serialnumber=lambda x: x["serialnumber"].astype(str).str.strip())
    .drop_duplicates(subset=["serialnumber"])
    .reset_index(drop=True)
)

bins_lat = bins["lat"].to_numpy()
bins_lon = bins["lon"].to_numpy()
bins_serial = bins["serialnumber"].to_numpy()

# create a list that contains for every bin geography metadata 
rows = []
for sn, lat, lon in zip(bins_serial, bins_lat, bins_lon):
    # Geoapify
    commercial_in_radius = geoapify_count_places(
        lat=lat, lon=lon, radius_m=radius_m,
        categories=categories, api_key=GEOAPIFY_API_KEY, limit=limit
    )

    # other bins in radius (exclude itself)
    d = haversine_vec(lat, lon, bins_lat, bins_lon)
    other_bins_in_radius = int(((d <= radius_m) & (bins_serial != sn)).sum())

    # ratio feature (name suggestion: "commercial_pressure")
    commercial_pressure = commercial_in_radius / (other_bins_in_radius + 1e-3)

    rows.append({
        "serialnumber": sn,
        "commercial_in_radius": commercial_in_radius,
        "other_bins_in_radius": other_bins_in_radius,
        "commercial_pressure": commercial_pressure
    })

    time.sleep(sleep_s)

features_df_radius_100 = pd.DataFrame(rows)

# MERGE BACK INTO df (by serialnumber)
df = df.assign(serialnumber=df["serialnumber"].astype(str).str.strip()).merge(
    features_df_radius_100,
    on="serialnumber",
    how="left"
)

display(df.head(9))

## Training XGBoost

In [ ]:
# %pip install xgboost scikit-learn

# YOU MIGHT HAVE TO RESTART THE JUPYTER KERNEL AFTER YOU INSTALLED XG BOOST
# %pip install pipreqs
# !pipreqs . --force

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

### Check for duplicates and remove them 

In [ ]:
unique_df = df.drop(columns=["coordinates"]).drop_duplicates()

dropped_count = len(df) - len(unique_df)
print(f"Rows dropped: {dropped_count}")

#actual dropping
df = df.drop(columns=["coordinates"]).drop_duplicates()

## Remove columns we dont want to use for training

In [ ]:
#DONT REMOVE TIMESTAMP FOR NOW BECAUSE WE NEED IT FOR SPLITTING THE DATASET INTO TRAINING AND TESTING 
df = df.drop(columns=["year"])

# Training a XGBoosted Tree

## Add lagged features (Weather and Fullness)
We do this after splitting so theres no dataleakage between train and test data

In [ ]:
# --- Daily features for fill dynamics (leakage-safe) ---
df = df.sort_values(["serialnumber", "timestamp"])
g = df.groupby("serialnumber")

# lags for weather + level (useful context)
for col in ["rain", "sunhours", "latestfullness"]:
    df[f"{col}_lag_1"] = g[col].shift(1)
    df[f"{col}_lag_2"] = g[col].shift(2)
    df[f"{col}_lag_3"] = g[col].shift(3)

for col in ["rain", "sunhours"]:
    for k in [1, 2, 3]:
        df[f"{col}_lead_{k}"] = g[col].shift(-k)
        
prev1 = df["latestfullness_lag_1"]

# pickup between yesterday -> today (today dropped vs yesterday)
df["pickup"] = (df["latestfullness"] < prev1).fillna(False).astype(int)

# daily growth (if pickup happened, treat yesterday end as 0 -> growth = today)
df["fill_rate"] = np.where(
    df["pickup"] == 1,
    df["latestfullness"],
    df["latestfullness"] - prev1
)

df["fill_rate"] = df["fill_rate"].clip(lower=0)

# fill_rate lags (past growth)
for k in [1, 2, 3, 7, 14]:
    df[f"fill_rate_lag_{k}"] = g["fill_rate"].shift(k)

# rolling averages (use past only via shift(1))
df["fill_rate_ma_3"] = g["fill_rate"].rolling(3).mean().reset_index(level=0, drop=True) # shift ????? yes/no
df["fill_rate_ma_7"]  = g["fill_rate"].rolling(7).mean().reset_index(level=0, drop=True)
df["fill_rate_ma_14"] = g["fill_rate"].rolling(14).mean().reset_index(level=0, drop=True) # shift ????? yes/no
df["fullness_ma_7"]   = g["latestfullness"].rolling(7).mean().reset_index(level=0, drop=True)
df["fullness_ma_14"]  = g["latestfullness"].rolling(14).mean().reset_index(level=0, drop=True)

# (optional) volatility signal
df["fill_rate_std_3"] = g["fill_rate"].rolling(3).std().reset_index(level=0, drop=True)
df["fill_rate_std_7"] = g["fill_rate"].rolling(7).std().reset_index(level=0, drop=True)

# target = next day's fullness per bin
df["target"] = g["latestfullness"].shift(-1)

# drop rows where we can't train (no target or no prev day)
df = df.dropna(subset=["target", "latestfullness_lag_1"])
df["serialnumber"] = df["serialnumber"].astype(int)

 

display(
    df[
        ["timestamp", "target", "fill_rate", "fill_rate_ma_7", "fill_rate_ma_14",
         "latestfullness", "latestfullness_lag_1", "fullness_ma_7", "pickup"]
    ].head(20)
)



In [ ]:
df

In [ ]:
# one-hot encode serialnumber
bin_dummies = pd.get_dummies(df["serialnumber"], prefix="bin")

# add to dataframe
df = pd.concat([df, bin_dummies], axis=1)



## Split the data given a timestamp

In [ ]:
# --- Feature/target split + time split + feature selection (updated for new features, no leakage) ---

cutoff_date = "2020-05-26"
mask = df["timestamp"] < cutoff_date

# target
y = df["fill_rate"]

bin_features = [c for c in df.columns if c.startswith("bin_")]
# add bin_features to keep_features if you want to add one-hot-encoded serialnumbers
# not used for now

# features (all are "known by end of yesterday" or exogenous)
keep_features = [
    # calendar / static
    "day_of_year",
    "day_of_week",
    "commercial_in_radius",
    "other_bins_in_radius",
    "commercial_pressure",
    # weather lags
    "rain_lag_1", "rain_lag_2", "rain_lag_3",
    "sunhours_lag_1", "sunhours_lag_2", "sunhours_lag_3",

    # state/history (lagged / rolling)
    # "latestfullness_lag_1", "latestfullness_lag_2", "latestfullness_lag_3",
    # "fullness_ma_7", "fullness_ma_14",
    "fill_rate_lag_1", "fill_rate_lag_2", "fill_rate_lag_3", "fill_rate_lag_7", "fill_rate_lag_14",
    "fill_rate_ma_7", "fill_rate_ma_14", "fill_rate_std_7",
    
    "rain_lead_1", "rain_lead_2", "rain_lead_3",
    "sunhours_lead_1", "sunhours_lead_2", "sunhours_lead_3",

    # optional: indicator that a pickup happened today (can help explain resets)
    # "pickup",
]

# build X and drop rows with missing feature values (from lags/rolling windows)
X = df.loc[:, keep_features].copy()
valid = X.notna().all(axis=1) & y.notna()

X, y = X.loc[valid], y.loc[valid]
mask = mask.loc[valid]

# time-based split
X_train, X_test = X.loc[mask], X.loc[~mask]
y_train, y_test = y.loc[mask], y.loc[~mask]

display(X_train.head())



## Training

In [ ]:
display(X_train)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# XGBoost for non-negative fill_rate
model = XGBRegressor(
    objective="reg:squarederror", # hier anderen error probieren wegen > 0
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
    # early_stopping_rounds=50,
)

train_pred = model.predict(X_train)
test_pred  = model.predict(X_test)
# train_pred = np.clip(model.predict(X_train), 0, None)
# test_pred  = np.clip(model.predict(X_test),  0, None)

# METRICS
mae  = mean_absolute_error(y_test, test_pred)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))
r2   = r2_score(y_test, test_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

y_mean = y_train.mean()
baseline_pred = np.full(len(y_test), y_mean)

baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2   = r2_score(y_test, baseline_pred)

print("Baseline (mean) MAE :", baseline_mae)
print("Baseline (mean) RMSE:", baseline_rmse)
print("Baseline (mean) R²  :", baseline_r2)


In [ ]:
print(X_test.columns)
print(X_test)

### Plot fill_rate of a random bin over time

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If df isn't loaded yet, load it (adjust path/parse_dates if needed)
# df = pd.read_csv("data/processed_filllevel.csv", parse_dates=["timestamp"])

# Pick a random bin that has fill_rate values
bins = df.loc[df["fill_rate"].notna(), "serialnumber"].unique()
rand_bin = np.random.choice(bins)

df_bin = df[df["serialnumber"] == rand_bin].sort_values("timestamp")

plt.figure(figsize=(10, 4))
plt.plot(df_bin["timestamp"], df_bin["fill_rate"])
plt.title(f"Fill rate over time — bin {rand_bin}")
plt.xlabel("Date")
plt.ylabel("Fill rate (daily growth)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pick a random bin with data
bins = df.loc[df["fill_rate"].notna(), "serialnumber"].unique()
rand_bin = np.random.choice(bins)

df_bin = (
    df[df["serialnumber"] == rand_bin]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ---- get all valid Mondays ----
df_bin["weekday"] = df_bin["timestamp"].dt.weekday  # Monday = 0
mondays = df_bin.loc[df_bin["weekday"] == 0, "timestamp"]

# keep Mondays that allow full 14-day window
mondays = mondays[
    mondays + pd.Timedelta(days=13) <= df_bin["timestamp"].max()
]

start_date = mondays.sample(1).iloc[0]
end_date = start_date + pd.Timedelta(days=13)

df_window = df_bin[
    (df_bin["timestamp"] >= start_date) &
    (df_bin["timestamp"] <= end_date)
]

# ---- plot ----
plt.figure(figsize=(10, 4))
plt.plot(df_window["timestamp"], df_window["fill_rate"])
plt.title(
    f"Fill rate — bin {rand_bin}\n"
    f"{start_date.date()} → {end_date.date()} (Mon–Sun ×2)"
)
plt.xlabel("Date")
plt.ylabel("Fill rate")
plt.tight_layout()
plt.show()


In [ ]:
# remove duplicate columns (safe)
df = df.loc[:, ~df.columns.duplicated()].copy()


df["y_next_fill_rate"] = df.groupby("serialnumber")["fill_rate"].shift(-1)


# ---- Time features ----
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["day_of_week"] = df["timestamp"].dt.dayofweek

# ---- Feature selection ----
base_features = [
    "commercial_in_radius",
    "other_bins_in_radius",
    "commercial_pressure",
    "fill_rate",
    "fill_rate_ma_3",
    "fill_rate_ma_7",
    "fill_rate_std_3",
    "fill_rate_std_7",
    "day_of_week",
    "rain",
    "sunhours",
    "rain_lead_1", "rain_lead_2", "rain_lead_3",
    "sunhours_lead_1", "sunhours_lead_2", "sunhours_lead_3",
]

bin_features = [c for c in df.columns if c.startswith("bin_")]
feature_cols = list(dict.fromkeys(base_features)) # neccessarry?

# ---- Ensure numeric (IMPORTANT) ----
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# drop rows where target OR features are missing
df = df.dropna(subset=["y_next_fill_rate"] + feature_cols).reset_index(drop=True)
print(df.columns)
# ---- Build LSTM sequences ----
SEQ_LEN = 30
X_list, y_list = [], []

for _, grp in df.groupby("serialnumber", sort=False):
    grp = grp.sort_values("timestamp")

    X_vals = grp[feature_cols].to_numpy(np.float32)
    y_vals = grp["y_next_fill_rate"].to_numpy(np.float32)

    if len(grp) <= SEQ_LEN:
        continue

    for i in range(SEQ_LEN, len(grp)):
        X_list.append(X_vals[i-SEQ_LEN:i])   # (SEQ_LEN, n_features)
        y_list.append(y_vals[i])

X = np.array(X_list, dtype=np.float32)   # (N, SEQ_LEN, n_features)
y = np.array(y_list, dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

# sanity check (this MUST pass)
assert X.ndim == 3


In [ ]:
TEST_FRAC = 0.2
split = int((1 - TEST_FRAC) * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler()

# flatten → scale → reshape
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])

X_train = scaler_X.fit_transform(X_train_flat).reshape(X_train.shape)
X_test  = scaler_X.transform(X_test_flat).reshape(X_test.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(100, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(1)
])

model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss="mse"
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=80,                 # can be higher, early stopping will cut it
    batch_size=64,
    validation_split=0.2,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test, verbose=0).ravel()

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")


# TRY TO PLOT CUMMULATIVE FILL_RATE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------- CONFIG ----------
feature_cols = base_features
TARGET_COL = "y_next_fill_rate"
TEST_FRAC = 0.2

# --------- 1) Prep df ----------
df2 = df.copy()
df2["timestamp"] = pd.to_datetime(df2["timestamp"])
df2 = df2.sort_values(["serialnumber", "timestamp"])

df2 = df2.dropna(subset=feature_cols + [TARGET_COL, "latestfullness", "fill_rate"]).reset_index(drop=True)

# --------- 2) Build sequences + meta ----------
X_list, y_list, meta = [], [], []

for sn, g in df2.groupby("serialnumber", sort=False):
    g = g.sort_values("timestamp").reset_index(drop=True)
    Xg = g[feature_cols].to_numpy(np.float32)
    yg = g[TARGET_COL].to_numpy(np.float32)

    if len(g) <= SEQ_LEN:
        continue

    for t in range(SEQ_LEN, len(g)):
        X_list.append(Xg[t-SEQ_LEN:t])
        y_list.append(yg[t])
        meta.append({"serialnumber": sn, "timestamp": g.loc[t, "timestamp"]})

X_all = np.array(X_list, dtype=np.float32)
y_all = np.array(y_list, dtype=np.float32)
meta_df = pd.DataFrame(meta)

# --------- 3) Test split ----------
# ---- per-bin split indices (keeps all bins in test) ----
train_idx, test_idx = [], []

for sn, g in meta_df.groupby("serialnumber", sort=False):
    idx = g.index.to_numpy()                  # indices into X_all/y_all
    cut = int((1 - TEST_FRAC) * len(idx))
    train_idx.append(idx[:cut])
    test_idx.append(idx[cut:])

train_idx = np.concatenate(train_idx)
test_idx  = np.concatenate(test_idx)

X_train_seq = X_all[train_idx]
y_train_seq = y_all[train_idx]
meta_train  = meta_df.iloc[train_idx].reset_index(drop=True)

X_test_seq  = X_all[test_idx]
y_test_seq  = y_all[test_idx]
meta_test   = meta_df.iloc[test_idx].reset_index(drop=True)


# scale with your existing scaler_X
X_test_flat = X_test_seq.reshape(-1, X_test_seq.shape[-1])
X_test_scaled = scaler_X.transform(X_test_flat).reshape(X_test_seq.shape)

# --------- 4) Predict ----------
y_pred = model.predict(X_test_scaled, verbose=0).ravel()

test_pred_df = meta_test.copy()
test_pred_df["y_true"] = y_test_seq
test_pred_df["y_pred"] = y_pred
test_pred_df = test_pred_df.sort_values(["timestamp", "serialnumber"]).reset_index(drop=True)

# --------- 5) Pick a 7-day window with MAX bin coverage ----------
daily_bins = (
    test_pred_df.groupby("timestamp")["serialnumber"]
    .nunique()
    .sort_index()
)

# require 7 consecutive timestamps (daily data assumed)
roll = daily_bins.rolling(7).sum()
best_start = roll.idxmax()
best_end = best_start + pd.Timedelta(days=6)

week = test_pred_df[(test_pred_df["timestamp"] >= best_start) & (test_pred_df["timestamp"] <= best_end)].copy()
week = week.sort_values(["serialnumber", "timestamp"]).reset_index(drop=True)

print("Chosen week:", best_start.date(), "→", best_end.date())
print("Unique bins in that week:", week["serialnumber"].nunique())

# choose RANDOM bin that appears in that best-coverage week
bins_in_week = week["serialnumber"].unique()
chosen_bin = np.random.choice(bins_in_week)

week_bin = (
    week[week["serialnumber"] == chosen_bin]
    .sort_values("timestamp")
    .head(7)
    .copy()
)

# --------- 6) Starting fill level ----------
df_bin = df2[df2["serialnumber"] == chosen_bin].sort_values("timestamp")
first_day = week_bin["timestamp"].iloc[0]

hist_before = df_bin[df_bin["timestamp"] < first_day]
start_level = float(hist_before["latestfullness"].iloc[-1]) if len(hist_before) else float(df_bin["latestfullness"].iloc[0])

# cumulative predicted fill level
pred_levels = start_level + np.cumsum(week_bin["y_pred"].to_numpy(float))

# --------- 7) Pretty weekday labels ----------
week_bin["weekday"] = week_bin["timestamp"].dt.day_name()
xlabels = week_bin["weekday"] + "\n" + week_bin["timestamp"].dt.strftime("%Y-%m-%d")

# --------- 8) Plot ----------
plt.figure(figsize=(10,4))
plt.plot(range(len(pred_levels)), pred_levels, marker="o", label="Predicted fill level")
plt.axhline(10, color="red", linestyle="--", label="Full (10)")

plt.title(f"Predicted fill level — best-coverage test week — bin {chosen_bin}")
plt.xlabel("Weekday / Date")
plt.ylabel("Fill level")
plt.xticks(range(len(xlabels)), xlabels, rotation=0)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assumes you already have: df2, test_pred_df from the previous cell

# ---- choose which bins to plot ----
bins_to_plot = [1511221, 1511210, 1510830, 1511197]

# --- Pick a 7-day window in TEST with MAX bin coverage ---
daily_bins = (
    test_pred_df.groupby("timestamp")["serialnumber"]
    .nunique()
    .sort_index()
)

roll = daily_bins.rolling(7).sum()
best_start = roll.idxmax()
best_end = best_start + pd.Timedelta(days=6)

week = test_pred_df[(test_pred_df["timestamp"] >= best_start) & (test_pred_df["timestamp"] <= best_end)].copy()
week = week.sort_values(["serialnumber", "timestamp"]).reset_index(drop=True)

# keep only requested bins (and only those actually present)
present = sorted(set(week["serialnumber"].unique()))
bins_in_week = [sn for sn in bins_to_plot if sn in present]

print("Chosen week:", best_start.date(), "→", best_end.date())
print("Requested bins:", bins_to_plot)
print("Bins present in this week:", bins_in_week)
print("Missing (not in week/test):", [sn for sn in bins_to_plot if sn not in present])

# --- weekday labels for x-axis (shared across all lines) ---
week_days = pd.date_range(best_start, best_end, freq="D")
xlabels = week_days.day_name() + "\n" + week_days.strftime("%Y-%m-%d")

# --- Plot selected bins (use index 0..6 so all bins align) ---
plt.figure(figsize=(12,5))

for sn in bins_in_week:
    week_bin = week[week["serialnumber"] == sn].sort_values("timestamp").copy()
    if week_bin.empty:
        continue

    # ensure exactly 7 days in this window (align by timestamp)
    week_bin = week_bin.set_index("timestamp").reindex(week_days).reset_index().rename(columns={"index":"timestamp"})
    week_bin = week_bin.dropna(subset=["y_pred"])  # drop missing days if any

    if week_bin.empty:
        continue

    # start level = latestfullness before first day of this bin's week
    df_bin = df2[df2["serialnumber"] == sn].sort_values("timestamp")
    first_day = week_days[0]
    hist_before = df_bin[df_bin["timestamp"] < first_day]
    start_level = float(hist_before["latestfullness"].iloc[-1]) if len(hist_before) else float(df_bin["latestfullness"].iloc[0])

    pred_levels = start_level + np.cumsum(week_bin["y_pred"].to_numpy(float))

    # x positions = 0..len-1 for clean weekday ticks
    x = [week_days.get_loc(ts) for ts in pd.to_datetime(week_bin["timestamp"])]
    plt.plot(x, pred_levels, marker="o", linewidth=1.5, alpha=0.9, label=str(sn))

plt.axhline(10, color="red", linestyle="--", label="Full (10)")
plt.title("Predicted fill level — best-coverage test week — selected bins")
plt.xlabel("Weekday / Date")
plt.ylabel("Fill level")
plt.xticks(range(len(week_days)), xlabels, rotation=0)
plt.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# Unique bins in the TEST dataset (from your sequence/meta test dataframe)

n_unique = test_pred_df["serialnumber"].nunique()
bins = sorted(test_pred_df["serialnumber"].unique())

print("Unique bins in TEST:", n_unique)
print("Bins:", bins)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# --- make sure we have the bin-id aligned with X/y after `valid` filtering ---
sn = df.loc[valid, "serialnumber"].astype(int)      # bin id per row
sn_test = sn.loc[~mask]                             # test rows only

# predictions you already computed
# test_pred = model.predict(X_test)

# --- per-bin metrics on TEST set ---
rows = []
for bin_id, idx in sn_test.groupby(sn_test).groups.items():
    yt = y_test.loc[idx]
    yp = pd.Series(test_pred, index=y_test.index).loc[idx]

    rows.append({
        "bin": int(bin_id),
        "n": int(len(idx)),
        "MAE":  mean_absolute_error(yt, yp),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "R2":   r2_score(yt, yp) if len(yt) > 1 else np.nan
    })

per_bin_metrics = pd.DataFrame(rows).sort_values(["MAE", "RMSE"])
display(per_bin_metrics)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rebuild a test dataframe with timestamp + serialnumber + actual + predicted
test_df = df.loc[X_test.index].copy()
test_df = test_df.sort_values("timestamp")
test_df["Predicted"] = y_pred        # y_pred from model.predict(X_test)

# 1️⃣ Pick a random bin (serialnumber)
bin_ids = test_df["serialnumber"].unique()
chosen_bin = np.random.choice(bin_ids)
print("Chosen bin:", chosen_bin)

# 2️⃣ Filter for this bin
bin_df = test_df[test_df["serialnumber"] == chosen_bin].sort_values("timestamp")

plt.figure(figsize=(10,4))
plt.plot(bin_df["timestamp"], bin_df["target"], label="Actual (next day)")
plt.plot(bin_df["timestamp"], bin_df["Predicted"], label="Predicted (next day)")
plt.title(f"Bin {chosen_bin} — Actual vs Predicted Next-Day Fullness")
plt.xlabel("Time")
plt.ylabel("Fill level")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(results["Actual"], label="Actual")
plt.plot(results["Predicted"], label="Predicted")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Actual vs Predicted – Time Series")
plt.legend()
plt.show()


In [ ]:
# Build a test dataframe with timestamp + actual + predicted
test_df = df.loc[X_test.index].copy()
test_df = test_df.sort_values("timestamp")
test_df["Predicted"] = y_pred

# Group by timestamp to compute averages
avg_df = test_df.groupby("timestamp").agg({
    "target": "mean",
    "Predicted": "mean"
}).reset_index()

# Plot averages
plt.figure(figsize=(10,4))
plt.plot(avg_df["timestamp"], avg_df["target"], label="Average Actual (next day)")
plt.plot(avg_df["timestamp"], avg_df["Predicted"], label="Average Predicted (next day)")
plt.xlabel("Time")
plt.ylabel("Fill level")
plt.title("Average Next-Day Fill Level — Actual vs Predicted")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
import pandas as pd

# Get feature importances as a DataFrame
importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

# Show top 10
print(importances)

In [ ]:
import matplotlib.pyplot as plt

print("Amount of datapoints: " , len(df["latestfullness"]))
print("List of all unique values found in the labels (latestfullness)" , df["latestfullness"].unique())

plt.hist(df["latestfullness"], bins=10)
plt.xlabel("latestfullness")
plt.ylabel("Count")
plt.title("Histogram")
plt.show()

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# ----- Convert regression target → 3-class labels -----
# adjust thresholds to your logic
def fullness_to_class(x):
    if x <= 3:
        return 0  # empty
    elif x <= 6:
        return 1  # medium
    else:
        return 2  # full

y_train_cls = y_train.apply(fullness_to_class)
y_test_cls  = y_test.apply(fullness_to_class)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# ----- Base classifier (no hard-coded n_estimators etc.) -----
base_clf = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    tree_method="hist",
    random_state=42
)

# ----- Reasonable grid -----
param_grid = {
    "n_estimators":   [100, 200],
    "max_depth":      [4, 6, 8],
    "learning_rate":  [0.03, 0.1],
    "subsample":      [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

# time-aware CV (no shuffling)
tscv = TimeSeriesSplit(n_splits=3)

grid = GridSearchCV(
    estimator=base_clf,
    param_grid=param_grid,
    cv=tscv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train_cls)

print("Best params:", grid.best_params_)
clf = grid.best_estimator_


In [ ]:
clf = grid.best_estimator_

# ----- Predict with best model -----
y_pred_cls = clf.predict(X_test)

# ----- Print confusion + classification report -----
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_cls, y_pred_cls))

print("\nClassification Report:")
print(classification_report(y_test_cls, y_pred_cls, target_names=["empty","medium","full"]))


In [ ]:

import matplotlib.pyplot as plt

# one plot per bin
for sn, df in test_df.sort_values("timestamp").groupby("serialnumber"):
    plt.figure(figsize=(8, 4))
    plt.plot(df["timestamp"], df["latestfullness"])
    plt.title(f"Bin {sn}")
    plt.xlabel("Timestamp")
    plt.ylabel("latestfullness")
    plt.tight_layout()
    plt.show()

## TRY LSTM


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler

# --------------------
# CONFIG
# --------------------
TIME_COL = "timestamp"
ID_COL = "serialnumber"
TARGET = "target"   # <--- predict target now

FEATURES = [
    "latestfullness",
    "latestfullness_lag_1", "latestfullness_lag_2", "latestfullness_lag_3",
    "fill_rate_1", "fill_rate_2",
    "day_of_week", "is_weekend",
    "sunhours_lag_1", "sunhours_lag_2", "sunhours_lag_3",
    # add your Geoapify/bin-radius features if present:
    # "commercial_in_radius","other_bins_in_radius","commercial_pressure",
]

SEQ_LEN = 7
BATCH_SIZE = 128
EPOCHS = 25

# --------------------
# 1) CLEAN
# --------------------
df_lstm = df.copy()
df_lstm[TIME_COL] = pd.to_datetime(df_lstm[TIME_COL], errors="coerce")
df_lstm[ID_COL] = df_lstm[ID_COL].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

df_lstm = df_lstm.dropna(subset=[TIME_COL, TARGET]).sort_values([ID_COL, TIME_COL])

# Fill NaNs in features (lags cause NaNs at the start)
df_lstm[FEATURES] = (
    df_lstm.groupby(ID_COL)[FEATURES]
    .apply(lambda g: g.ffill().bfill())
    .reset_index(level=0, drop=True)
).fillna(0)

# Target: if it's continuous, keep float; if it's binary, you can cast to int
y_is_binary = set(pd.unique(df_lstm[TARGET].dropna())) <= {0, 1, 0.0, 1.0}
if y_is_binary:
    df_lstm[TARGET] = df_lstm[TARGET].astype(int)
else:
    df_lstm[TARGET] = df_lstm[TARGET].astype(float)

# --------------------
# 2) TIME-BASED SPLIT (recommended)
# --------------------
cutoff = df_lstm[TIME_COL].quantile(0.8)
train_df = df_lstm[df_lstm[TIME_COL] <= cutoff].copy()
val_df   = df_lstm[df_lstm[TIME_COL] >  cutoff].copy()

# --------------------
# 3) SCALE FEATURES (fit on train only)
# --------------------
scaler = StandardScaler()
train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])
val_df[FEATURES]   = scaler.transform(val_df[FEATURES])

# --------------------
# 4) MAKE SEQUENCES PER BIN
# --------------------
def make_sequences(frame, seq_len):
    X_list, y_list = [], []
    for _, g in frame.sort_values([ID_COL, TIME_COL]).groupby(ID_COL):
        Xg = g[FEATURES].to_numpy(dtype=np.float32)
        yg = g[TARGET].to_numpy(dtype=np.float32)

        if len(g) <= seq_len:
            continue

        for t in range(seq_len, len(g)):
            X_list.append(Xg[t-seq_len:t])
            y_list.append(yg[t])

    if not X_list:
        return np.empty((0, seq_len, len(FEATURES)), dtype=np.float32), np.empty((0,), dtype=np.float32)

    return np.stack(X_list), np.array(y_list, dtype=np.float32)

X_train, y_train = make_sequences(train_df, SEQ_LEN)
X_val,   y_val   = make_sequences(val_df,   SEQ_LEN)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)

# --------------------
# 5) MODEL
# --------------------
n_features = len(FEATURES)

model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, n_features)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    # output depends on target type:
    layers.Dense(1, activation="sigmoid" if y_is_binary else "linear")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy" if y_is_binary else "mse",
    metrics=(["accuracy", tf.keras.metrics.AUC(name="auc")] if y_is_binary else ["mae"])
)

model.summary()

# --------------------
# 6) TRAIN
# --------------------
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)


In [ ]:
# --- TEST / EVALUATE THE TRAINED LSTM (run after training) ---

import numpy as np
import matplotlib.pyplot as plt

# 1) Evaluate on validation/test set
scores = model.evaluate(X_val, y_val, verbose=0)
print(dict(zip(model.metrics_names, scores)))

# 2) Predict
y_pred = model.predict(X_val, verbose=0).ravel()

# 3) Metrics + plots (binary vs regression)
if y_is_binary:
    from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

    auc = roc_auc_score(y_val, y_pred)
    print("AUC:", auc)

    y_hat = (y_pred >= 0.5).astype(int)
    print("Confusion matrix:\n", confusion_matrix(y_val, y_hat))
    print("\nClassification report:\n", classification_report(y_val, y_hat, digits=3))

    # Probability histogram
    plt.figure()
    plt.hist(y_pred, bins=30)
    plt.title("Predicted probabilities (val)")
    plt.xlabel("p(target=1)")
    plt.ylabel("count")
    plt.show()

else:
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

    # Pred vs true scatter
    plt.figure()
    plt.scatter(y_val, y_pred, s=10)
    plt.title("Predicted vs true (val)")
    plt.xlabel("true target")
    plt.ylabel("pred target")
    plt.show()


In [ ]:
display(df)

In [ ]:
SEQ_LEN = 6
TEST_FRAC = 0.2

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []

idx = 0
for serial, g in df.groupby("serialnumber"):
    n = max(0, len(g) - SEQ_LEN)
    if n == 0:
        continue

    split_i = int((1 - TEST_FRAC) * n)

    X_train_list.append(X[idx:idx+split_i])
    y_train_list.append(y[idx:idx+split_i])

    X_test_list.append(X[idx+split_i:idx+n])
    y_test_list.append(y[idx+split_i:idx+n])

    idx += n

X_train = np.concatenate(X_train_list, axis=0)
X_test  = np.concatenate(X_test_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
y_test  = np.concatenate(y_test_list, axis=0)


print(X_train.shape)
print(X_train)

In [ ]:
# LSTM training + prediction (configured like the paper: stacked LSTM + Dense output)
# Assumes you already have: X_train, y_train, X_test, y_test from the previous cell

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# --- model (stacked LSTM) ---
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64),
    Dense(1)
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse"
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=70,
    verbose=1
)

# --- predict ---
y_pred = model.predict(X_test).reshape(-1)

# quick metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
